# 01: Confirm the event and check satellite coverage

**Goal of this notebook:** before building anything, make sure the data can answer the question.

1. Confirm the storm in satellite rainfall (GPM IMERG).
2. List Sentinel-1 radar passes before and after the storm.
3. List reasonably clear Sentinel-2 images.
4. Measure how much cropland is in the study area (ESA WorldCover).

If there is no Sentinel-1 pass shortly before **and** after the storm from the **same orbit**, the before/after comparison won't work, and we need to rethink before going further.

In [ ]:
# Setup: run once per session
# !pip install earthengine-api geemap pandas matplotlib   # uncomment on Colab if needed
import ee
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = "YOUR-EARTH-ENGINE-PROJECT-ID"   # <- change this
ee.Authenticate()
ee.Initialize(project=PROJECT)

## Configuration

`EVENT_DATE` is the day the storm hit (local date). Fill it in from the original phone photos, news reports or family.

`HOME_LAT`, `HOME_LON` are optional: your village's location, used for a smaller study area around it. Leave as `None` to use the whole district.

In [ ]:
EVENT_DATE = None        # e.g. "2026-10-05"  <- REQUIRED
WINDOW_DAYS = 24         # days before and after the event to search
HOME_LAT, HOME_LON = None, None   # optional, decimal degrees
LOCAL_RADIUS_KM = 15     # radius of the local study area around home

if EVENT_DATE is None:
    raise ValueError("Set EVENT_DATE first (YYYY-MM-DD).")

event = ee.Date(EVENT_DATE)
start = event.advance(-WINDOW_DAYS, "day")
end = event.advance(WINDOW_DAYS + 1, "day")

## Study area

District boundary from FAO GAUL. If the filter returns nothing, the name may be spelled differently in the dataset: the cell prints the Uttarakhand district names so you can fix it.

In [ ]:
gaul = ee.FeatureCollection("FAO/GAUL/2015/level2")
uk = gaul.filter(ee.Filter.eq("ADM1_NAME", "Uttarakhand"))
district = uk.filter(ee.Filter.eq("ADM2_NAME", "Udham Singh Nagar"))

if district.size().getInfo() == 0:
    print("District not found. Names in the dataset:")
    print(sorted(uk.aggregate_array("ADM2_NAME").getInfo()))
else:
    district_geom = district.geometry()
    if HOME_LAT is not None:
        aoi = ee.Geometry.Point([HOME_LON, HOME_LAT]).buffer(LOCAL_RADIUS_KM * 1000)
        print(f"Using a {LOCAL_RADIUS_KM} km area around home.")
    else:
        aoi = district_geom
        print("Using the whole district.")
    print("Study area:", round(aoi.area(maxError=100).divide(1e6).getInfo()), "km²")

## 1. Rainfall around the event (GPM IMERG)

IMERG gives rainfall rate (mm/hr) every 30 minutes, so each image × 0.5 = mm in that half hour. Days here are **UTC**; India is UTC+5:30, so an evening storm may fall on the next UTC day.

**Note:** if the bars around the event are missing, the IMERG collection in Earth Engine may not cover such recent dates yet. In that case, rely on IMD bulletins and news reports for timing, and come back to this later.

In [ ]:
imerg = ee.ImageCollection("NASA/GPM_L3/IMERG_V07").select("precipitation")
n_days = 2 * WINDOW_DAYS + 1

def daily_rain(i):
    d = start.advance(ee.Number(i), "day")
    day = imerg.filterDate(d, d.advance(1, "day"))
    total = day.sum().multiply(0.5)   # mm/hr x 0.5 hr per image
    mean = total.reduceRegion(ee.Reducer.mean(), aoi, 11132, maxPixels=1e9)
    return ee.Feature(None, {
        "date": d.format("YYYY-MM-dd"),
        "n_images": day.size(),
        "rain_mm": ee.Dictionary(mean).get("precipitation", -1),
    })

rain = ee.FeatureCollection(ee.List.sequence(0, n_days - 1).map(daily_rain))
df_rain = pd.DataFrame([f["properties"] for f in rain.getInfo()["features"]])
df_rain.loc[df_rain.n_images == 0, "rain_mm"] = float("nan")   # no data that day
df_rain["date"] = pd.to_datetime(df_rain["date"])

ax = df_rain.plot(x="date", y="rain_mm", kind="bar", figsize=(12, 3), legend=False)
ax.set_ylabel("Rainfall (mm/day, area mean)")
ax.set_title("Daily rainfall around the event (GPM IMERG, UTC days)")
plt.tight_layout(); plt.show()
print("Days with no IMERG data:", int(df_rain.rain_mm.isna().sum()))

## 2. Sentinel-1 radar passes

For a fair before/after comparison, both images should come from the **same orbit direction and relative orbit**, so the satellite looks at the ground from the same angle. Look for a pair that brackets the event.

In [ ]:
s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
      .filterBounds(aoi)
      .filterDate(start, end)
      .filter(ee.Filter.eq("instrumentMode", "IW"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))

def s1_row(img):
    return ee.Feature(None, {
        "date": img.date().format("YYYY-MM-dd HH:mm"),
        "platform": img.get("platform_number"),
        "pass": img.get("orbitProperties_pass"),
        "relative_orbit": img.get("relativeOrbitNumber_start"),
    })

feats = s1.map(s1_row).getInfo()["features"]
df_s1 = pd.DataFrame([f["properties"] for f in feats])
if df_s1.empty:
    print("No Sentinel-1 images found in this window.")
else:
    df_s1 = df_s1.sort_values("date").drop_duplicates().reset_index(drop=True)
    df_s1["when"] = df_s1["date"].apply(lambda d: "before" if d[:10] < EVENT_DATE else "after")
    display(df_s1)
    pairs = df_s1.groupby(["pass", "relative_orbit"])["when"].agg(lambda w: set(w))
    usable = pairs[pairs.apply(lambda w: {"before", "after"} <= w)]
    print("Orbit tracks with images both before and after:", list(usable.index))

## 3. Sentinel-2 optical images

Clouds are common after storms. These help show crop condition on clear days before and after.

In [ ]:
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(aoi)
      .filterDate(start, end)
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40)))

def s2_row(img):
    return ee.Feature(None, {
        "date": img.date().format("YYYY-MM-dd"),
        "cloud_pct": img.get("CLOUDY_PIXEL_PERCENTAGE"),
    })

feats = s2.map(s2_row).getInfo()["features"]
df_s2 = pd.DataFrame([f["properties"] for f in feats])
if df_s2.empty:
    print("No Sentinel-2 images under 40% cloud in this window.")
else:
    df_s2 = df_s2.sort_values("date").reset_index(drop=True)
    display(df_s2)

## 4. Cropland in the study area (ESA WorldCover)

WorldCover class 40 is cropland. This mask keeps the analysis on farmland. WorldCover is from 2021, so check it against recent imagery where it matters.

In [ ]:
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()
cropland = worldcover.eq(40).selfMask()
crop_area = (cropland.multiply(ee.Image.pixelArea())
             .reduceRegion(ee.Reducer.sum(), aoi, 30, maxPixels=1e10)
             .get("Map"))
print("Cropland:", round(ee.Number(crop_area).divide(1e6).getInfo()), "km²")

## 5. Look at it

Shows the study area, the cropland mask and the first Sentinel-1 image after the event (VH, in dB).

In [ ]:
import geemap
m = geemap.Map()
m.centerObject(aoi, 10)
m.addLayer(cropland, {"palette": ["#f2c94c"]}, "Cropland (WorldCover)")
s1_after = s1.filterDate(event, end).sort("system:time_start")
if s1_after.size().getInfo() > 0:
    m.addLayer(s1_after.first().select("VH").clip(aoi), {"min": -25, "max": -5}, "Sentinel-1 VH after")
m.addLayer(ee.FeatureCollection([ee.Feature(aoi)]).style(color="red", fillColor="00000000"), {}, "Study area")
m

## Before moving to Phase 2, check

- [ ] Event date and time confirmed (from photos, news, IMD or family), and visible in rainfall if data exists.
- [ ] At least one orbit track with Sentinel-1 images both before and after the event.
- [ ] Gap between those images is short enough that crops didn't change much for other reasons (harvest, growth).
- [ ] Cropland mask looks right around your village.
- [ ] Ground truth collection has started (see `data/ground_truth/`).

Log what you found, including problems, in `notes.md`.